# Exploratory Analysis

### **Reading compras.csv**

In [0]:
# Read the CSV file, skipping the header
with open('/Volumes/workspace/default/prueba_bavaria/compras.csv', 'r') as f:
    header = f.readline()
    rows = f.readlines()

# Replace | with , in the rows
rows = [row.replace('|', ',') for row in rows]

# Write the header and modified rows to a new CSV file
output_path = '/Volumes/workspace/default/prueba_bavaria/compras_clean.csv'
with open(output_path, 'w') as f:
    f.write(header)
    f.writelines(rows)

# Read the cleaned CSV into a Spark DataFrame
df_compras = spark.read.format("csv").option("header", "true").load(output_path)

# Create Databricks SQL table
df_compras.write.format("delta").mode("overwrite").saveAsTable("workspace.default.prueba_bavaria_compras")

In [0]:
display(df_compras)
df_compras.printSchema()

### **Reading productos.csv**

In [0]:
# Read the CSV file, skipping the header
with open('/Volumes/workspace/default/prueba_bavaria/productos.csv', 'r') as f:
    header = f.readline()
    rows = f.readlines()

# Replace | with , in the rows
rows = [row.replace('|', ',') for row in rows]

# Write the header and modified rows to a new CSV file
output_path = '/Volumes/workspace/default/prueba_bavaria/productos_clean.csv'
with open(output_path, 'w') as f:
    f.write(header)
    f.writelines(rows)

# Read the cleaned CSV into a Spark DataFrame
df_productos = spark.read.format("csv").option("header", "true").load(output_path)

# Create Databricks SQL table
df_productos.write.format("delta").mode("overwrite").saveAsTable("workspace.default.prueba_bavaria_productos")

In [0]:
display(df_productos)
df_productos.printSchema()

### **Join Tables**

In [0]:
df_compras = spark.table("workspace.default.prueba_bavaria_compras")
df_productos = spark.table("workspace.default.prueba_bavaria_productos")

df = df_compras.join(df_productos, on="SKU_ID", how="left")

In [0]:
from pyspark.sql.functions import col, to_date

df = df.withColumn("GMV", col("GMV").cast("double")) \
       .withColumn("Unit_Quantity", col("Unit_Quantity").cast("int")) \
       .withColumn("Volume", col("Volume").cast("double")) \
       .withColumn("SKU SIZE", col("SKU SIZE").cast("double")) \
       .withColumn("Order_Date", to_date("Order_Date"))

In [0]:
df.printSchema()
df.describe().show()

In [0]:
df.filter(col("Category").isNull()).count()

### **Business Metrics**

In [0]:
from pyspark.sql.functions import countDistinct, sum

df.select(
    countDistinct("Customer").alias("customers"),
    countDistinct("POC").alias("sales_points"),
    sum("GMV").alias("total_sales"),
    sum("Unit_Quantity").alias("total_units")
).show()

#Customer Profiles

### **Features by Customer**

In [0]:
from pyspark.sql.functions import avg, countDistinct

customer_df = df.groupBy("Customer").agg(
    countDistinct("Order_Date").alias("frequency"),
    sum("GMV").alias("total_spend"),
    avg("GMV").alias("avg_ticket"),
    sum("Unit_Quantity").alias("total_volume"),
    countDistinct("Brand").alias("brand_diversity"),
    countDistinct("POC").alias("point_diversity")
)

### **K-Means Preparation**

In [0]:
from pyspark.ml.feature import VectorAssembler

features = [
    "frequency",
    "total_spend",
    "avg_ticket",
    "total_volume",
    "brand_diversity",
    "point_diversity"
]

assembler = VectorAssembler(inputCols=features, outputCol="features")

customer_vector = assembler.transform(customer_df).select("Customer", "features")

### **Scaling for Clustering**

In [0]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaled_features")

scaler_model = scaler.fit(customer_vector)
customer_scaled = scaler_model.transform(customer_vector)

### **K-Means Application**

In [0]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans(k=4, seed=42, featuresCol="scaled_features")
model = kmeans.fit(customer_scaled)

clusters = model.transform(customer_scaled)

In [0]:
final_df = clusters.join(customer_df, on="Customer")

final_df.groupBy("prediction").agg(
    avg("frequency"),
    avg("total_spend"),
    avg("avg_ticket"),
    avg("total_volume"),
    avg("brand_diversity")
).show()

###Profile Interpretation

**Cluster 2 — High-Value Customers**

Frequency: very high

Total spend: extremely high

Ticket: high

Volume: very high

Brand diversity: very high

Insight:
This segment represents the highest-value customers, with high frequency and spending, plus broad brand exploration.

They are probably bars, distributors, or wholesale customers.
They buy many brands
They have high purchasing power


**Cluster 3 — Frequent Multi-Brand Customers**

Frequency: high

Spend: high

Ticket: medium

Diversity: high

Insight:
Frequent customers with relevant spending, characterized by high brand diversity.

They buy often, but are not loyal
Sensitive to promotions and availability


**Cluster 1 — Medium Customers**

Frequency: medium

Spend: medium

Ticket: high

Diversity: medium

Insight:
Intermediate segment with balanced behavior between frequency and spending.

They are the core of the business
Growth potential

**Cluster 0 — Occasional Customers**

Frequency: very low

Spend: low

Ticket: medium

Diversity: low

Insight:
Low frequency and low value customers, with sporadic purchases.

Low engagement
Possible churn

In [0]:
from pyspark.sql.functions import sum as spark_sum

df.groupBy("Order_Date").agg(spark_sum(col("GMV").cast("double")))

#Charts

### **Revenue Concentration**

In [0]:
import pandas as pd
import numpy as np

pdf = final_df.select("Customer", "total_spend").toPandas()
pdf = pdf.sort_values(by="total_spend", ascending=False)

pdf["cum_gmv"] = pdf["total_spend"].cumsum()
pdf["cum_perc_gmv"] = pdf["cum_gmv"] / pdf["total_spend"].sum()
pdf["cum_perc_clients"] = np.arange(1, len(pdf)+1) / len(pdf)

import matplotlib.pyplot as plt

plt.figure()
plt.plot(pdf["cum_perc_clients"], pdf["cum_perc_gmv"])
plt.xlabel("Percentage of customers")
plt.ylabel("Cumulative percentage of GMV")
plt.title("Revenue Concentration (Pareto Curve)")
plt.show()

### **Compare Segments**

In [0]:
cluster_pd = final_df.groupBy("prediction").agg(
    avg("frequency").alias("frequency"),
    avg("total_spend").alias("spend"),
    avg("avg_ticket").alias("ticket"),
    avg("total_volume").alias("volume")
).toPandas()

cluster_pd.set_index("prediction").plot(kind="bar")
plt.title("Customer profile by cluster")
plt.ylabel("Average values")
plt.xticks(rotation=0)
plt.show()

In [0]:
%sql
SELECT * FROM workspace.default.prueba_bavaria_compras

Databricks visualization. Run in Databricks to view.